# Data and monitoring schedule
Inspect training history only. The benchmark generator is unchanged. Truth is an evaluation table, never a predictor. This notebook creates inputs only if the chosen run does not already exist.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
RUN = Path(os.environ.get('ANOMALY_RUN', str(ROOT / 'outputs' / 'v4_run'))).resolve()
CONFIG = ROOT / 'configs' / 'industry_agnostic.yaml'
from anomaly_detection.pipeline import prepare, canonical, bounds, load


In [ ]:
RUN, config = prepare(CONFIG, RUN)
start, train_end, dev_end, end = bounds(config)
data, expected, registry = canonical(RUN, train_end)
display(pd.DataFrame([dict(metric=k, **v.to_dict()) for k,v in registry.items()]))
display(data.groupby(["metric_name", "quality"]).size().rename("rows").to_frame())

In [ ]:
manifest = json.loads((RUN / "input_manifest.json").read_text())
display({k: manifest[k] for k in ["input_rows", "truth_rows", "boundaries", "test_status", "healthy_training_verified"]})
from anomaly_detection.evaluation import exposure
print("Scheduled training entity-days:", exposure(expected, start, train_end))

In [ ]:
data.pivot_table(index="event_time", columns="metric_name", values="value").plot(figsize=(10,4), title="Training power telemetry: population means for exploration only");

Population means above are descriptive; fitting remains local. Unknown errors are not zeros. Missing and late observations retain quality reasons and scheduled exposure. Positive scale floors, units and direction come from the registry.